# Monetary EXIOBASE walkthrough

This notebook is the practical walkthrough for parsing monetary EXIOBASE in MARIO.

It focuses on the two main cases:

- monetary IOT
- monetary SUT


## What this notebook covers

- which parser entry points to use
- what local dataset layout MARIO expects
- how to download supported monetary releases
- how to parse one monetary IOT
- how to parse one monetary SUT
- how to add SUT extensions from the matching IOT when available

## Relevant Zenodo releases

- [v3.8.2 (IOTs and SUTs available)](https://doi.org/10.5281/zenodo.5589597)
- [v3.9.4 (IOTs only)](https://doi.org/10.5281/zenodo.14614930)
- [v3.9.5 (IOTs only)](https://doi.org/10.5281/zenodo.14869924)
- [v3.9.6 (IOTs only)](https://doi.org/10.5281/zenodo.15689391)
- [v3.10.1 (IOTs only)](https://doi.org/10.5281/zenodo.18937492)

## Main entry point

For normal user workflows, the public entry points are:

- `mario.parse_exiobase(...)`


## Key arguments

The key public arguments are:

- `table`: choose `"IOT"` or `"SUT"`
- `unit`: use `"Monetary"`
- `path`: local EXIOBASE directory to parse
- `year`: optional metadata override, usually inferred from the payload
- `add_extensions`: monetary SUT-only helper to import extensions from a matching IOT


## Local layout expectation

The current monetary parsers expect the EXIOBASE dataset to be available as a local directory with the files required by the corresponding release. The parser auto-detects the layout and database version from the directory contents and metadata.

For SUT parsing, extensions can be added by pointing `add_extensions` to the matching monetary IOT directory.


In [1]:
import mario

## Optional download step

If you do not already have the monetary dataset locally, MARIO can download the supported monetary releases directly from Zenodo.

For IOT releases, you must specify the system (`"ixi"` or `"pxp"`).


In [2]:
download_iot = mario.download_exiobase3(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2',
    years=[2019],
    table="IOT",
    system="ixi",
    version="3.8.2",
)

The SUT download path is currently available only for **3.8.2**.


In [3]:
download_sut = mario.download_exiobase3(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2',
    years=[2019],
    table="SUT",
    version="3.8.2",
)

## Parse a monetary IOT

Use this path when you have a monetary EXIOBASE input-output table already available locally.


In [4]:
db_iot = mario.parse_exiobase(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2/IOT_2019_ixi',
    table="IOT",
    unit="Monetary",
)

INFO Parser: reading EXIOBASE IOT from /path/to/exiobase/iot_directory.


INFO Parser: reading EXIOBASE IOT extensions from /path/to/exiobase/iot_directory.


INFO Parser: using bundled legacy satellite layout.


INFO Parser: EXIOBASE IOT parsed with 163 sectors, 9 value-added rows and 1104 extension rows.


INFO Metadata: initialized.


## Parse a monetary SUT

Use this path when the source dataset is the monetary supply-use table.

At the moment, this path is tied to EXIOBASE **3.8.2**.


In [5]:
db_sut = mario.parse_exiobase(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2/MRSUT_2019',
    table="SUT",
    unit="Monetary",
)

INFO Parser: reading EXIOBASE SUT from /path/to/exiobase/sut_directory.


INFO Parser: no matching IOT provided; using empty satellite extensions.


INFO Parser: EXIOBASE SUT parsed with 163 activities, 200 commodities, 12 value-added rows.


INFO Metadata: initialized.


## Add SUT extensions from the matching monetary IOT

If you have both the monetary SUT and the corresponding monetary IOT, MARIO can read the extension blocks from the IOT and attach them to the SUT parse.


In [6]:
db_eesut = mario.parse_exiobase(
    path='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2/MRSUT_2019',
    table="SUT",
    unit="Monetary",
    add_extensions='/Volumes/SanDisk SSD/Work/MARIO/database/EXIOBASE/3.8.2/IOT_2019_ixi',
)

INFO Parser: reading EXIOBASE SUT from /path/to/exiobase/sut_directory.


INFO Parser: importing satellite extensions from matching IOT at /path/to/exiobase/iot_directory.


INFO Parser: reading EXIOBASE IOT extensions from /path/to/exiobase/iot_directory.


INFO Parser: using bundled legacy satellite layout.


INFO Parser: EXIOBASE SUT parsed with 163 activities, 200 commodities, 12 value-added rows.


INFO Metadata: initialized.


## Inspect the parsed database

Once parsed, the result is a standard MARIO database. From here the normal exploration methods apply.

A simple way to inspect a parsed MARIO database is to evaluate the object name in a notebook cell (for example, `db_iot` or `db_sut`).  
This prints a compact summary of its core properties.


In [7]:
db_iot

name = exio382_ntnu
table = IOT
scenarios = ['baseline']
Factor of production = 9
Satellite account = 1104
Consumption category = 7
Region = 49
Sector = 163

In [8]:
db_sut

name = MRSUT_2019
table = SUT
tech_assumption = industry-based
scenarios = ['baseline']
Activity = 163
Commodity = 200
Factor of production = 12
Satellite account = 1
Consumption category = 7
Region = 49

One may then check the list of sectors, activities, commodities or any other set of the databse

In [9]:
db_iot.sectors

['Cultivation of paddy rice',
 'Cultivation of wheat',
 'Cultivation of cereal grains nec',
 'Cultivation of vegetables, fruit, nuts',
 'Cultivation of oil seeds',
 'Cultivation of sugar cane, sugar beet',
 'Cultivation of plant-based fibers',
 'Cultivation of crops nec',
 'Cattle farming',
 'Pigs farming',
 'Poultry farming',
 'Meat animals nec',
 'Animal products nec',
 'Raw milk',
 'Wool, silk-worm cocoons',
 'Manure treatment (conventional), storage and land application',
 'Manure treatment (biogas), storage and land application',
 'Forestry, logging and related service activities (02)',
 'Fishing, operating of fish hatcheries and fish farms; service activities incidental to fishing (05)',
 'Mining of coal and lignite; extraction of peat (10)',
 'Extraction of crude petroleum and services related to crude oil extraction, excluding surveying',
 'Extraction of natural gas and services related to natural gas extraction, excluding surveying',
 'Extraction, liquefaction, and regasificat

In [10]:
db_sut.commodities

['Paddy rice',
 'Wheat',
 'Cereal grains nec',
 'Vegetables, fruit, nuts',
 'Oil seeds',
 'Sugar cane, sugar beet',
 'Plant-based fibers',
 'Crops nec',
 'Cattle',
 'Pigs',
 'Poultry',
 'Meat animals nec',
 'Animal products nec',
 'Raw milk',
 'Wool, silk-worm cocoons',
 'Manure (conventional treatment)',
 'Manure (biogas treatment)',
 'Products of forestry, logging and related services (02)',
 'Fish and other fishing products; services incidental of fishing (05)',
 'Anthracite',
 'Coking Coal',
 'Other Bituminous Coal',
 'Sub-Bituminous Coal',
 'Patent Fuel',
 'Lignite/Brown Coal',
 'BKB/Peat Briquettes',
 'Peat',
 'Crude petroleum and services related to crude oil extraction, excluding surveying',
 'Natural gas and services related to natural gas extraction, excluding surveying',
 'Natural Gas Liquids',
 'Other Hydrocarbons',
 'Uranium and thorium ores (12)',
 'Iron ores',
 'Copper ores and concentrates',
 'Nickel ores and concentrates',
 'Aluminium ores and concentrates',
 'Precious

For deeper and more targeted search, the search function can be used to explore specific text in specific sets

In [11]:
db_iot.search("sectors","electricity")

['Production of electricity by coal',
 'Production of electricity by gas',
 'Production of electricity by nuclear',
 'Production of electricity by hydro',
 'Production of electricity by wind',
 'Production of electricity by petroleum and other oil derivatives',
 'Production of electricity by biomass and waste',
 'Production of electricity by solar photovoltaic',
 'Production of electricity by solar thermal',
 'Production of electricity by tide, wave, ocean',
 'Production of electricity by Geothermal',
 'Production of electricity nec',
 'Transmission of electricity',
 'Distribution and trade of electricity']

In [12]:
db_sut.search("commodities","gas")

['Manure (biogas treatment)',
 'Natural gas and services related to natural gas extraction, excluding surveying',
 'Natural Gas Liquids',
 'Gas Coke',
 'Motor Gasoline',
 'Aviation Gasoline',
 'Gasoline Type Jet Fuel',
 'Gas/Diesel Oil',
 'Refinery Gas',
 'Liquefied Petroleum Gases (LPG)',
 'Biogasoline',
 'Electricity by gas',
 'Coke oven gas',
 'Blast Furnace Gas',
 'Oxygen Steel Furnace Gas',
 'Gas Works Gas',
 'Biogas',
 'Distribution services of gaseous fuels through mains',
 'Food waste for treatment: biogasification and land application',
 'Paper waste for treatment: biogasification and land application',
 'Sewage sludge for treatment: biogasification and land application']

## Notes

- prefer the dispatcher when you want one stable public API;
- derived matrices are not materialized eagerly unless you ask for them;
- if you need a monetary SUT, use the 3.8.2 release rather than the newer monetary IOT-only releases.
